In [ ]:
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together

# Install compatible versions of langchain-core and langchain-openai
%pip install langchain-community==0.4.1
%pip install langchain-text-splitters==1.0.0
%pip install langchain-openai==1.1.0
%pip install langsmith==0.4.49
%pip install langchain==1.1.0

# Install remaining packages
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.2.1
%pip install PyPDF2==3.0.1 -q --user
%pip install rank_bm25==0.2.2
%pip install langchain-classic==1.0.0
%pip install langchain_core==1.1.3

# New installs for document loaders
%pip install beautifulsoup4==4.14.3  
%pip install python-docx==1.2.0
%pip install docx2txt==0.9
%pip install jq==1.10.0

In [ ]:
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_chroma import Chroma
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from PyPDF2 import PdfReader
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [ ]:
# variables
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']
embedding_function = OpenAIEmbeddings()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
pdf_path = "google-2023-environmental-report.pdf"
collection_name = "google_environmental_report"
str_output_parser = StrOutputParser()
user_query = "What are Google's environmental initiatives?"

In [ ]:

#### DOCUMENT LOADERS ####

In [ ]:
# set up our file to be available in all formats:
from bs4 import BeautifulSoup
import docx
import json

# Document paths
pdf_path = "google-2023-environmental-report.pdf"
html_path = "google-2023-environmental-report.html"
word_path = "google-2023-environmental-report.docx"
json_path = "google-2023-environmental-report.json"

with open(pdf_path, "rb") as pdf_file:
    pdf_reader = PdfReader(pdf_file)
    pdf_text = "".join(page.extract_text() for page in pdf_reader.pages)

    # Text to HTML
    soup = BeautifulSoup("<html><body></body></html>", "html.parser")
    soup.body.append(pdf_text)
    with open(html_path, "w", encoding="utf-8") as html_file:
        html_file.write(str(soup))

    # Text to Word
    doc = docx.Document()
    doc.add_paragraph(pdf_text)
    doc.save(word_path)

    # Text to JSON
    with open(json_path, "w") as json_file:
        json.dump({"text": pdf_text}, json_file)

In [ ]:
#### INDEXING ####

In [ ]:
# HTML Loader
# Other options:
# https://python.langchain.com/v0.2/docs/how_to/document_loader_html/
from langchain_community.document_loaders import BSHTMLLoader

loader = BSHTMLLoader(html_path)
docs = loader.load()

In [ ]:
# PDF Loader
# Other options:
# https://python.langchain.com/v0.2/docs/how_to/document_loader_pdf/
from PyPDF2 import PdfReader

docs = []
with open(pdf_path, "rb") as pdf_file:
    pdf_reader = PdfReader(pdf_file)
    pdf_text = "".join(page.extract_text() for page in pdf_reader.pages)
    docs = [Document(page_content=page) for page in pdf_text.split("\n\n")]

In [ ]:
# Microsoft Word Loader
# Other options:
# https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader(word_path)
docs = loader.load()

In [ ]:
# JSON Loader
# https://python.langchain.com/v0.2/docs/how_to/document_loader_json/
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path=json_path,
    jq_schema='.text',
)

docs = loader.load()

In [ ]:
# use the same splitter for all of them:
character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=200
)

splits = character_splitter.split_documents(docs)

In [ ]:
dense_documents = [Document(page_content=doc.page_content, 
    metadata={**doc.metadata, "id": str(i), "search_source": "dense"}) for 
        i, doc in enumerate(splits)]
sparse_documents = [Document(page_content=doc.page_content, 
    metadata={**doc.metadata, "id": str(i), "search_source": "sparse"}) for 
        i, doc in enumerate(splits)]

In [ ]:
# Chroma Vector Store
chroma_client = chromadb.Client()
vectorstore = Chroma.from_documents(
    documents=dense_documents,
    embedding=embedding_function,
    collection_name=collection_name,
    client=chroma_client
)

In [ ]:
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
sparse_retriever = BM25Retriever.from_documents(sparse_documents, k=10)
ensemble_retriever = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5], c=0, k=10)

In [ ]:
#### RETRIEVAL and GENERATION ####